# stdio와 원격 전송·권한 경계

로컬 stdio 서버와 원격 Streamable HTTP 서버는 실행 주체·비밀 전달·권한 확인의 경계가 어떻게 다를까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. 비밀 값은 출력하지 않습니다.

## 패키지 버전 확인

패키지 버전 확인에 필요한 값과 결과를 확인합니다.

In [1]:
# 패키지 버전 확인에 필요한 값을 준비하고 결과를 확인합니다.
%pip install -q "langchain[mcp]==1.4.0" langchain-openai==1.6.0 fastmcp==4.0.3 mcp==2.2.0 python-dotenv==1.2.3
import warnings
from importlib.metadata import version
from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", message=r".*langchain\.mcp.*", category=LangChainBetaWarning)

for package in ["langchain", "langchain-openai", "fastmcp", "mcp"]:
    print(package, version(package))

Note: you may need to restart the kernel to use updated packages.


langchain 1.4.0
langchain-openai 1.6.0
fastmcp 4.0.3
mcp 2.2.0


## Before: 전체 환경 전달 확인

Before: 전체 환경 전달 확인에 필요한 값과 결과를 확인합니다.

### 코드 읽기와 비교 경계

stdio 연결에서는 클라이언트가 서버 명령을 하위 프로세스로 시작하고 표준 입력·출력으로 JSON-RPC 메시지를 교환합니다. `env`에 넘긴 사전은 그 프로세스가 볼 수 있는 환경 변수를 정합니다. 부모 프로세스의 전체 환경을 복사하면 API 키처럼 서버에 필요하지 않은 값도 함께 전달될 수 있습니다. 필요한 변수만 새 사전에 넣는 After 코드는 서버 기능은 유지하면서 노출 범위를 줄입니다.

Streamable HTTP에서는 서버가 별도 주소에서 실행되므로 프로세스 환경 대신 네트워크 요청의 인증 정보와 서버 측 인가가 중요합니다. TLS는 전송 중 내용을 보호하고 인증은 요청 주체를 확인하지만, 어떤 사용자가 어떤 도구를 실행할 수 있는지는 인가 규칙이 정합니다. stdio가 로컬이라는 이유로 안전하다고 단정할 수도 없습니다. 하위 프로세스는 전달받은 운영체제 권한으로 파일과 네트워크에 접근할 수 있으므로 실행 파일, 환경, 작업 폴더를 함께 제한해야 합니다.


In [2]:
# Before: 전체 환경 전달 확인에 필요한 값을 준비하고 결과를 확인합니다.
import os

unsafe_env = dict(os.environ)
print("전체 환경 변수 개수:", len(unsafe_env))
print("OPENAI_API_KEY 이름 포함:", "OPENAI_API_KEY" in unsafe_env)
assert len(unsafe_env) > 2

전체 환경 변수 개수: 60
OPENAI_API_KEY 이름 포함: False


## After: 서버에 필요한 환경만 선택

After: 서버에 필요한 환경만 선택에 필요한 값과 결과를 확인합니다.

In [3]:
# After: 서버에 필요한 환경만 선택에 필요한 값을 준비하고 결과를 확인합니다.
safe_env = {
    "PATH": os.environ.get("PATH", ""),
    "PYTHONUNBUFFERED": "1",
}
print("전달할 이름:", sorted(safe_env))
print("OPENAI_API_KEY 이름 포함:", "OPENAI_API_KEY" in safe_env)
assert "OPENAI_API_KEY" not in safe_env

전달할 이름: ['PATH', 'PYTHONUNBUFFERED']
OPENAI_API_KEY 이름 포함: False


## 제한한 환경으로 실제 stdio 서버 호출

제한한 환경으로 실제 stdio 서버 호출에 필요한 값과 결과를 확인합니다.

In [4]:
# 제한한 환경으로 실제 stdio 서버 호출에 필요한 값을 준비하고 결과를 확인합니다.
import sys
from pathlib import Path
from langchain.mcp import MCPAdapter

relative = Path("labs/part6/ch28-mcp/course_server.py")
server_path = next(folder / relative for folder in [Path.cwd(), *Path.cwd().parents] if (folder / relative).is_file())
server_config = {"mcpServers": {"course": {
    "command": sys.executable,
    "args": [str(server_path)],
    "env": safe_env,
}}}

async def call_over_stdio() -> tuple[list[str], object]:
    async with MCPAdapter(server_config) as adapter:
        tools = await adapter.list_tools()
        result = await tools[0].ainvoke({"topic": "운영"})
        return [tool.name for tool in tools], result

tool_names, result = await call_over_stdio()
print("transport: stdio")
print("tools:", tool_names)
print("result:", result)
assert tool_names == ["lookup_policy"]
assert "20:00" in str(result)

transport: stdio
tools: ['lookup_policy']
result: [{'type': 'text', 'text': '평일은 09:00~20:00, 토요일은 10:00~17:00에 운영합니다.', 'id': 'lc_33b10a9c-0ae4-4dda-96d9-032b233a16e9'}]


## 전송별 신뢰 경계 비교

전송별 신뢰 경계 비교에 필요한 값과 결과를 확인합니다.

In [5]:
# 전송별 신뢰 경계 비교에 필요한 값을 준비하고 결과를 확인합니다.
boundaries = [
    ("stdio", "클라이언트가 로컬 프로세스 시작", "선택한 환경 변수", "운영체제 사용자 권한"),
    ("Streamable HTTP", "원격 서비스가 별도 실행", "TLS와 Authorization", "사용자·도구별 인가"),
]
for row in boundaries:
    print(" | ".join(row))
assert boundaries[0][0] != boundaries[1][0]

stdio | 클라이언트가 로컬 프로세스 시작 | 선택한 환경 변수 | 운영체제 사용자 권한
Streamable HTTP | 원격 서비스가 별도 실행 | TLS와 Authorization | 사용자·도구별 인가
